# Agente Inteligente de Gestión de Inventario
## Evaluación Parcial 2 - ISY0101

Este notebook demuestra el funcionamiento interno del agente paso a paso, mostrando el uso de las herramientas, la memoria y la orquestación. Es ideal para que el docente pueda evaluar los componentes individuales (Clean Architecture).

In [7]:
import os
import sys
from dotenv import load_dotenv

# Añadir el directorio 'backend' al path para resolver las importaciones de 'src'
backend_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'backend'))
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

# Cargar variables de entorno reales desde la raíz
env_path = os.path.abspath(os.path.join(os.getcwd(), '..', '.env'))
if os.path.exists(env_path):
    load_dotenv(env_path)
    print("Entorno cargado desde archivo .env real")
else:
    load_dotenv(os.path.abspath(os.path.join(os.getcwd(), '..', '.env.example')))
    print("Entorno cargado desde archivo .env.example (fallback)")

print(f"Google API Key presente: {'Sí' if os.getenv('GOOGLE_API_KEY') else 'No'}")
print(f"GitHub Token presente: {'Sí' if os.getenv('GITHUB_TOKEN') else 'No'}")

Entorno cargado desde archivo .env real
Google API Key presente: Sí
GitHub Token presente: Sí


### 1. Probando el Proveedor LLM (Triple Fallback)

In [8]:
from src.infrastructure.llm_provider import TripleFallbackLLMProvider

provider = TripleFallbackLLMProvider()
respuesta = provider.generate_response("Hola, ¿qué productos tienen stock bajo?")
print(respuesta)

Hola, no tengo acceso a información en tiempo real sobre inventarios o stock de productos. Te recomendaría que consultes directamente con la tienda o el sistema de gestión de inventarios que estés utilizando para obtener esa información. Si necesitas ayuda con algo más, ¡estaré encantado de ayudarte!


### 2. Probando Herramientas Individuales (IE1)

In [9]:
from src.tools.inventory_query import consultar_inventario
from src.tools.weather_checker import consultar_clima

print("--- Consulta de Inventario ---")
print(consultar_inventario.invoke("SKU-1001"))

print("\n--- Consulta de Clima ---")
print(consultar_clima.invoke("Santiago"))

--- Consulta de Inventario ---

    Producto: Bloqueador Solar FPS 50 (Estacional)
    SKU: SKU-1001
    Stock Físico (Actual): 25
    Stock en Tránsito: 0
    Stock Total Disponible: 25
    Ubicación: Viña del Mar
    Lead Time (Días): 5
    

--- Consulta de Clima ---
Pronóstico para Santiago:
- 2026-06-06: 33°C (Soleado, ALERTA OLA DE CALOR)
- 2026-06-07: 35°C (Soleado, ALERTA OLA DE CALOR)
- 2026-06-08: 34°C (Soleado, ALERTA OLA DE CALOR)
- 2026-06-09: 30°C (Soleado)
- Resto de la semana: ~28°C



### 3. Recuperación Semántica (RAG) (IE4)

In [10]:
from src.memory.semantic_retriever import SemanticRetriever

# Nota: Asegúrate de haber ejecutado scripts/ingest_docs.py primero
try:
    retriever = SemanticRetriever()
    politica = retriever.retrieve_policies("¿Cuál es el stock de seguridad para productos estacionales en ola de calor?")
    print(politica)
except Exception as e:
    print(f"Asegúrate de ejecutar la ingesta de documentos: {e}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Contexto recuperado de políticas de OmniRetail:

[1] Contexto: # Política de Gestión de Inventario - OmniRetail S.A.

## 4. Alertas de Quiebre
Cualquier producto cuyo stock actual sea igual o menor al 50% de su stock de seguridad debe ser marcado como **CRÍTICO** y requiere reposición acelerada (express), sin importar el costo de envío adicional.

[2] Contexto: # Política de Gestión de Inventario - OmniRetail S.A.

## 1. Niveles de Stock de Seguridad
Para mitigar riesgos de quiebres de stock, OmniRetail establece los siguientes niveles de stock de seguridad obligatorios, basados en la categoría del producto:

*   **Productos Perecederos (Alimentos frescos):** Stock de seguridad de 2 días de venta promedio.
*   **Productos Estacionales (Verano/Invierno):** Stock de seguridad del 20% sobre la proyección de ventas para el mes en curso. Si hay alertas climáticas (ej. ola de calor), el stock de seguridad debe aumentar un 15% adicional.
*   **Electrónica y Electrodomésticos:** Stock de segur

### 4. Orquestación Completa con LangChain (IE5, IE6)

In [11]:
from src.application.agent import InventoryAgent

agent = InventoryAgent()
respuesta_agente = agent.process_request("Hay pronóstico de ola de calor en Santiago la próxima semana. Analiza el stock del SKU-1001 y recomiéndame si debo pedir más. Justifica con las políticas de la empresa y escribe un reporte.")

print("\n\n=== RESPUESTA DEL AGENTE ===\n")
print(respuesta_agente)

/home/hector/Escritorio/SolucionesIA/SolucionesIA/backend/src/memory/conversation_memory.py:26: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  return ConversationBufferWindowMemory(




> Entering new AgentExecutor chain...

Invoking: `consultar_inventario` with `{'sku': 'SKU-1001'}`



    Producto: Bloqueador Solar FPS 50 (Estacional)
    SKU: SKU-1001
    Stock Físico (Actual): 25
    Stock en Tránsito: 0
    Stock Total Disponible: 25
    Ubicación: Viña del Mar
    Lead Time (Días): 5
    
Invoking: `analizar_tendencias` with `{'sku': 'SKU-1001', 'dias': 30}`



    Análisis de Tendencias para Bloqueador Solar FPS 50 (SKU-1001)
    Período analizado: Últimos 30 días
    Total unidades vendidas: 608
    Venta diaria promedio: 20.27 unidades/día
    
Invoking: `consultar_clima` with `{'ciudad': 'Santiago'}`


Pronóstico para Santiago:
- 2026-06-06: 33°C (Soleado, ALERTA OLA DE CALOR)
- 2026-06-07: 35°C (Soleado, ALERTA OLA DE CALOR)
- 2026-06-08: 34°C (Soleado, ALERTA OLA DE CALOR)
- 2026-06-09: 30°C (Soleado)
- Resto de la semana: ~28°C

Invoking: `buscar_politicas_empresa` with `{'query': 'reposición de inventario'}`




Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Contexto recuperado de políticas de OmniRetail:

[1] Contexto: # Política de Gestión de Inventario - OmniRetail S.A.

## 4. Alertas de Quiebre
Cualquier producto cuyo stock actual sea igual o menor al 50% de su stock de seguridad debe ser marcado como **CRÍTICO** y requiere reposición acelerada (express), sin importar el costo de envío adicional.

[2] Contexto: # Política de Gestión de Inventario - OmniRetail S.A.

## 2. Puntos de Reorden (ROP)
El punto de reorden determina cuándo debe emitirse una nueva orden de compra al proveedor.
*   **Cálculo Estándar:** ROP = (Venta Diaria Promedio * Lead Time del Proveedor) + Stock de Seguridad.
*   **Regla de Acción:** Siempre que el inventario físico caiga por debajo del ROP, se debe generar automáticamente una sugerencia de reposición.


Invoking: `escribir_reporte` with `{'titulo': 'Recomendación de Reposición para SKU-1001', 'contenido': '### Análisis del SKU-1001 (Bloqueador Solar FPS 50)\n\n1. **Inventario Actual**:\n   - Stock Físico: 25